In [1]:
from pathlib import Path
import hashlib
import frontmatter
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [2]:
def parse_document(path):
    post = frontmatter.load(path)

    metadata = post.metadata

    return {
        "filename": path.name,
         # YAML metadata
        "title": metadata.get("title"),
        "category": metadata.get("category"),
        "audience": metadata.get("audience"),
        "topics": metadata.get("topics", []),
        "related": metadata.get("related", []),
        "last_updated": metadata.get("last_updated"),

        # Markdown body only
        "content": post.content,
    }

In [3]:
DATA_DIR = Path("data")



def load_markdown_files():

    documents = []

    # Loop through every Markdown file
    for path in DATA_DIR.glob("*.md"):

        # STEP 2 happens here
        document = parse_document(path)

        documents.append(document)

    return documents


In [4]:
documents = load_markdown_files()

In [5]:
len(documents)

6

In [6]:
documents[0]

{'filename': 'streaming.md',
 'title': 'Streaming',
 'category': 'Advanced Features',
 'audience': 'Developers',
 'topics': ['streaming', 'real-time responses', 'latency', 'user experience'],
 'related': ['responses-api.md', 'function-calling.md', 'models.md'],
 'last_updated': '2026-07',
 'content': '# Streaming\n\n## Overview\n\nStreaming allows an application to receive a model\'s output incrementally instead of waiting for the complete response. This improves perceived responsiveness and provides a better user experience for longer generations.\n\n## Why Use Streaming?\n\nStreaming is useful when:\n\n- Responses are long.\n- Low perceived latency is important.\n- Building chat applications.\n- Displaying generated text as it arrives.\n\n## Basic Example\n\n```python\nfrom openai import OpenAI\n\nclient = OpenAI()\n\nstream = client.responses.create(\n    model="gpt-5",\n    input="Write a short story about space.",\n    stream=True\n)\n\nfor event in stream:\n    print(event)\n```\

In [7]:
documents[1]

{'filename': 'responses-api.md',
 'title': 'Responses API',
 'category': 'Core API',
 'audience': 'Developers',
 'topics': ['responses api', 'text generation', 'multimodal', 'sdk', 'python'],
 'related': ['authentication.md',
  'streaming.md',
  'function-calling.md',
  'models.md'],
 'last_updated': '2026-07',
 'content': '# Responses API\n\n**Last Updated:** July 2026\n\n## Overview\n\nThe Responses API is the primary interface for interacting with OpenAI language models. It provides a consistent way to send inputs to a model and receive generated outputs. It supports text generation, structured outputs, multimodal inputs, streaming, and tool calling.\n\n## Basic Request\n\n```python\nfrom openai import OpenAI\n\nclient = OpenAI()\n\nresponse = client.responses.create(\n    model="gpt-5",\n    input="Explain what an API is in one paragraph."\n)\n\nprint(response.output_text)\n```\n\n## Request Components\n\n| Field | Description |\n|--------|-------------|\n| model | Model to use |\n

In [8]:
def make_chunk_id(filename, chunk_number):

    text = f"{filename}-{chunk_number}"

    return hashlib.md5(text.encode()).hexdigest()

In [9]:
HEADERS = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]


In [18]:
def chunk_documents(documents):
    """
    Split each document into smaller chunks while
    preserving metadata.
    """

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=HEADERS
    )

    chunks = []

    for document in documents:

        splits = splitter.split_text(document["content"])
        print(f"Split {document['filename']} into {len(splits)} chunks.")

        for i, split in enumerate(splits):

            chunk = {
                    "id": make_chunk_id(document["filename"], i),

                    "filename": document["filename"],
                    "title": document["title"],
                    "category": document["category"],
                    "audience": document["audience"],
                    "topics": document["topics"],
                    "related": document["related"],
                    "last_updated": document["last_updated"],

                    "header1": split.metadata.get("h1"),
                    "header2": split.metadata.get("h2"),
                    "header3": split.metadata.get("h3"),

                    "text": split.page_content,
                }

            chunks.append(chunk)

    return chunks

In [19]:
chunks = chunk_documents(documents)

Split streaming.md into 10 chunks.
Split responses-api.md into 13 chunks.
Split function-calling.md into 10 chunks.
Split authentication.md into 14 chunks.
Split models.md into 10 chunks.
Split embeddings.md into 11 chunks.


In [20]:
len(chunks)

68

In [21]:
chunks[0]

{'id': 'c3328abc23c57566db55a2819ec446a8',
 'filename': 'streaming.md',
 'title': 'Streaming',
 'category': 'Advanced Features',
 'audience': 'Developers',
 'topics': ['streaming', 'real-time responses', 'latency', 'user experience'],
 'related': ['responses-api.md', 'function-calling.md', 'models.md'],
 'last_updated': '2026-07',
 'header1': 'Streaming',
 'header2': 'Overview',
 'header3': None,
 'text': "Streaming allows an application to receive a model's output incrementally instead of waiting for the complete response. This improves perceived responsiveness and provides a better user experience for longer generations."}

In [22]:
chunks[1]

{'id': 'cbe4b9d749778bc721ad5c861e4ba775',
 'filename': 'streaming.md',
 'title': 'Streaming',
 'category': 'Advanced Features',
 'audience': 'Developers',
 'topics': ['streaming', 'real-time responses', 'latency', 'user experience'],
 'related': ['responses-api.md', 'function-calling.md', 'models.md'],
 'last_updated': '2026-07',
 'header1': 'Streaming',
 'header2': 'Why Use Streaming?',
 'header3': None,
 'text': 'Streaming is useful when:  \n- Responses are long.\n- Low perceived latency is important.\n- Building chat applications.\n- Displaying generated text as it arrives.'}

In [23]:
def prepare_texts(chunks):
    texts = []

    for chunk in chunks:
        embedding_text = f"""
Title: {chunk['title']}
Category: {chunk['category']}
Section: {chunk.get('header2') or chunk.get('header1')}

Content:
{chunk['text']}
""".strip()

        texts.append(embedding_text)

    return texts

In [24]:
prepared_texts = prepare_texts(chunks)

In [26]:
len(prepared_texts)

68

In [27]:
prepared_texts[0]

"Title: Streaming\nCategory: Advanced Features\nSection: Overview\n\nContent:\nStreaming allows an application to receive a model's output incrementally instead of waiting for the complete response. This improves perceived responsiveness and provides a better user experience for longer generations."